# Axe 5 — Télétravail et organisation

**Question DRH :** le mode de travail et le contexte de l’équipe sont-ils associés à la satisfaction ?

Population : personnes en emploi qui codent, France et pays frontaliers, tous secteurs. Les réponses sont volontaires. Les associations ne démontrent pas de causalité.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px

# Permet de lancer le notebook depuis notebooks/ ou la racine.
racine = Path.cwd()
if not (racine / "analyse").is_dir():
    racine = racine.parent
if not (racine / "analyse").is_dir():
    raise RuntimeError("Ouvrir le notebook depuis le dossier du projet.")
sys.path.insert(0, str(racine))

from analyse.chargement import charger
from analyse.population import preparer
from analyse.affichage import GRIS, ACCENT, ENCRE, SEUIL, legende_population

base = preparer(charger(2026))
df = base[base["Zone"].isin(["France", "Pays frontaliers"])].copy()
df["JobSat"] = pd.to_numeric(df["JobSat"], errors="coerce")
qui = "Développeurs en emploi en France et pays frontaliers, tous secteurs"
print(legende_population(df, qui, 2026))
df[["WorkSetting", "TechBudget", "OrgSize", "JobSat"]].head()

Développeurs en emploi en France et pays frontaliers, tous secteurs · 2026 · 3 376 répondants · volontaires, non représentatifs de tous les développeurs


,WorkSetting,TechBudget,OrgSize,JobSat
4,Remote,Decreased,100 to 499 employees,8.0
9,"Your choice (very flexible, you can come in wh...",I don't know,"1,000 to 4,999 employees",9.0
45,"Your choice (very flexible, you can come in wh...",Increased,"Just me - I am a freelancer, sole proprietor, ...",8.0
57,Remote,Stayed the same,100 to 499 employees,10.0
68,"Hybrid (some remote, leans heavy to in-person)",Stayed the same,Less than 20 employees,7.0


## 1. Vérifier les réponses et les valeurs manquantes

In [2]:
for colonne in ["WorkSetting", "TechBudget", "OrgSize", "JobSat"]:
    print("\n", colonne)
    print(df[colonne].value_counts(dropna=False))


 WorkSetting
WorkSetting
Remote                                                                          867
Hybrid (some remote, leans heavy to in-person)                                  849
Hybrid (some in-person, leans heavy to flexibility)                             670
Your choice (very flexible, you can come in when you want or just as needed)    552
In-person                                                                       435
NaN                                                                               3
Name: count, dtype: int64

 TechBudget
TechBudget
Stayed the same    1019
I don't know       1002
Decreased           687
Increased           652
NaN                  16
Name: count, dtype: int64

 OrgSize
OrgSize
20 to 99 employees                                    670
100 to 499 employees                                  645
Less than 20 employees                                563
1,000 to 4,999 employees                              424
10,000 or more employees

## 2. Satisfaction selon le mode de travail
On calcule la moyenne sur les réponses valides à JobSat (0 à 10) et à la question étudiée. Les catégories de moins de 50 réponses sont exclues des graphiques.

In [3]:
MODES = {
    "Remote": "Télétravail complet",
    "Hybrid (some remote, leans heavy to in-person)": "Hybride, surtout au bureau",
    "Hybrid (some in-person, leans heavy to flexibility)": "Hybride, surtout à distance",
    "Your choice (very flexible, you can come in when you want or just as needed)": "Au choix",
    "In-person": "Au bureau",
}
BUDGETS = {
    "Increased": "Augmenté",
    "Stayed the same": "Stable",
    "Decreased": "Baissé",
    "I don't know": "Ne sait pas",
}

def resume_satisfaction(colonne, traductions):
    valides = df.loc[df["JobSat"].between(0, 10) & df[colonne].isin(traductions)].copy()
    valides["categorie"] = valides[colonne].map(traductions)
    stats = (valides.groupby("categorie")["JobSat"]
             .agg(satisfaction="mean", repondants="size").reset_index())
    return valides, stats

valides_mode, modes = resume_satisfaction("WorkSetting", MODES)
modes

,categorie,satisfaction,repondants
0,Au bureau,7.009346,428
1,Au choix,7.121597,551
2,"Hybride, surtout au bureau",6.944379,845
3,"Hybride, surtout à distance",6.854135,665
4,Télétravail complet,7.145665,865


In [4]:
def graphique(valides, stats, titre, accents=()):
    visibles = stats[stats["repondants"] >= SEUIL].copy()
    if visibles.empty:
        print("Effectif trop faible")
        return
    visibles["etiquette"] = visibles["satisfaction"].map(lambda v: f"{v:.2f} / 10")
    fig = px.bar(visibles, x="satisfaction", y="categorie", orientation="h",
                 text="etiquette", hover_data=["repondants"], title=titre)
    fig.update_traces(marker_color=[ACCENT if x in accents else GRIS for x in visibles["categorie"]],
                      textposition="outside", cliponaxis=False)
    fig.update_layout(xaxis=dict(range=[0, 10], title="Satisfaction moyenne / 10", showgrid=False),
                      yaxis_title=None, plot_bgcolor="white", paper_bgcolor="white", font_color=ENCRE)
    fig.show()
    print(legende_population(valides[valides["categorie"].isin(visibles["categorie"])], qui, 2026))

def constat(stats):
    visibles = stats[stats["repondants"] >= SEUIL]
    if len(visibles) < 2:
        return "Effectif insuffisant pour comparer", None
    haut = visibles.loc[visibles["satisfaction"].idxmax()]
    bas = visibles.loc[visibles["satisfaction"].idxmin()]
    ecart = haut["satisfaction"] - bas["satisfaction"]
    return f"{haut['categorie']} : {haut['satisfaction']:.2f}/10 contre {bas['satisfaction']:.2f}/10 pour {bas['categorie']}", ecart

titre_mode, ecart_mode = constat(modes)
graphique(valides_mode, modes, titre_mode)
print("Je remarque :", titre_mode)
print("Je me demande si ces écarts persistent selon la taille de l'entreprise.")

Développeurs en emploi en France et pays frontaliers, tous secteurs · 2026 · 3 354 répondants · volontaires, non représentatifs de tous les développeurs
Je remarque : Télétravail complet : 7.15/10 contre 6.85/10 pour Hybride, surtout à distance
Je me demande si ces écarts persistent selon la taille de l'entreprise.


## 3. Satisfaction selon le budget de l’équipe

In [5]:
valides_budget, budgets = resume_satisfaction("TechBudget", BUDGETS)
budgets

,categorie,satisfaction,repondants
0,Augmenté,7.530864,648
1,Baissé,6.416422,682
2,Ne sait pas,6.894106,1001
3,Stable,7.200197,1014


In [6]:
titre_budget, ecart_budget = constat(budgets)
graphique(valides_budget, budgets, titre_budget, accents=("Baissé", "Augmenté"))
print("Je remarque :", titre_budget)
print("Je me demande si les écarts reflètent aussi les difficultés économiques des équipes.")

Développeurs en emploi en France et pays frontaliers, tous secteurs · 2026 · 3 345 répondants · volontaires, non représentatifs de tous les développeurs
Je remarque : Augmenté : 7.53/10 contre 6.42/10 pour Baissé
Je me demande si les écarts reflètent aussi les difficultés économiques des équipes.


## 4. Comparer les écarts observés

In [7]:
if ecart_mode is not None and ecart_budget is not None:
    print(f"Écart maximal selon le mode : {ecart_mode:.2f} point(s) sur 10")
    print(f"Écart maximal selon le budget : {ecart_budget:.2f} point(s) sur 10")
    if ecart_budget > ecart_mode:
        print("Je remarque : l'écart descriptif est plus important selon le budget.")
    elif ecart_budget < ecart_mode:
        print("Je remarque : l'écart descriptif est plus important selon le mode de travail.")
    else:
        print("Je remarque : les écarts descriptifs sont identiques.")
else:
    print("Comparaison impossible avec le seuil retenu.")
print("Limite : les populations de répondants peuvent différer entre les deux questions. Ces écarts ne prouvent pas un effet causal.")

Écart maximal selon le mode : 0.29 point(s) sur 10
Écart maximal selon le budget : 1.11 point(s) sur 10
Je remarque : l'écart descriptif est plus important selon le budget.
Limite : les populations de répondants peuvent différer entre les deux questions. Ces écarts ne prouvent pas un effet causal.


## 5. Préparer le filtre de taille d’entreprise
Vérifier les catégories réelles avant de les regrouper en moins de 100, 100 à 999 et 1 000 salariés ou plus.

In [8]:
df["OrgSize"].value_counts(dropna=False)

OrgSize
20 to 99 employees                                    670
100 to 499 employees                                  645
Less than 20 employees                                563
1,000 to 4,999 employees                              424
10,000 or more employees                              415
Just me - I am a freelancer, sole proprietor, etc.    254
500 to 999 employees                                  236
5,000 to 9,999 employees                              132
I don’t know                                           30
NaN                                                     7
Name: count, dtype: int64

## 6. Préparer la comparaison 2025–2026
Selon `HARMONISATION.md`, la population de 2025 doit être sélectionnée via `MainBranch` et `Employment`. On renomme `RemoteWork` en `WorkSetting`. Tous secteurs confondus uniquement ; la composition des échantillons peut changer.

In [9]:
from analyse.population import ajouter_zone, EN_EMPLOI

base25 = charger(2025)
codent25 = ["I am a developer by profession",
            "I am not primarily a developer, but I write code sometimes as part of my work/studies"]
df25 = ajouter_zone(base25[base25["MainBranch"].isin(codent25) & base25["Employment"].isin(EN_EMPLOI)].copy())
df25 = df25[df25["Zone"].isin(["France", "Pays frontaliers"])].rename(columns={"RemoteWork": "WorkSetting"})

resultats = []
for annee, source in [(2025, df25), (2026, df)]:
    valides = source[source["WorkSetting"].isin(MODES)]
    effectifs = valides["WorkSetting"].map(MODES).value_counts()
    if len(valides):
        for mode, nb in effectifs.items():
            if nb >= SEUIL:
                resultats.append({"annee": str(annee), "mode": mode, "repondants": nb, "part": 100 * nb / len(valides)})
    print(legende_population(valides, qui + " — réponses valides au mode de travail", annee))
comparaison = pd.DataFrame(resultats)
comparaison

Développeurs en emploi en France et pays frontaliers, tous secteurs — réponses valides au mode de travail · 2025 · 5 026 répondants · volontaires, non représentatifs de tous les développeurs
Développeurs en emploi en France et pays frontaliers, tous secteurs — réponses valides au mode de travail · 2026 · 3 373 répondants · volontaires, non représentatifs de tous les développeurs


,annee,mode,repondants,part
0,2025,"Hybride, surtout au bureau",1246,24.791086
1,2025,Télétravail complet,1157,23.020294
2,2025,"Hybride, surtout à distance",1151,22.900915
3,2025,Au choix,834,16.593713
4,2025,Au bureau,638,12.693991
5,2026,Télétravail complet,867,25.704121
6,2026,"Hybride, surtout au bureau",849,25.170471
7,2026,"Hybride, surtout à distance",670,19.863623
8,2026,Au choix,552,16.365253
9,2026,Au bureau,435,12.896531


In [10]:
if not comparaison.empty:
    fig = px.bar(comparaison, x="mode", y="part", color="annee", barmode="group",
                 color_discrete_map={"2025": GRIS, "2026": ACCENT},
                 hover_data=["repondants"], labels={"part": "Part (%)", "mode": "Mode de travail", "annee": "Année"})
    fig.update_layout(plot_bgcolor="white", paper_bgcolor="white", font_color=ENCRE)
    fig.show()
print("Les parts utilisent toutes les réponses valides de chaque année comme dénominateur ; les catégories de moins de 50 réponses ne sont pas affichées.")

Les parts utilisent toutes les réponses valides de chaque année comme dénominateur ; les catégories de moins de 50 réponses ne sont pas affichées.


## 7. Conclusion à compléter après exécution
- Quel mode présente la satisfaction la plus élevée ?
- Quelle est l’amplitude des écarts selon le budget et le mode ?
- Quels changements observe-t-on entre 2025 et 2026 ?
- Pour la DRH : quelles pistes méritent une investigation complémentaire ?

Ne pas conclure que le télétravail ou le budget cause une variation de satisfaction. L’enquête ne mesure pas les départs.